To get the Sender Type

In [3]:
import pandas as pd
import re

# Read Excel file
df = pd.read_excel('/content/emails.xlsx')

def get_sender_type(sender):
    match = re.search(r'<([^<>]+)>', str(sender))

    if match:
        email = match.group(1)
    else:
        email = str(sender).strip()

    if '@' in email:
        domain = email.split('@')[-1].lower().strip()

        if domain == 'arisecapital.in':
            return 'internal'

    return 'external'

df['sender_type'] = df['from'].apply(get_sender_type)

df.to_excel('/content/emails_sender_type.xlsx', index=False)

print("Processing completed!")
print("Updated file saved as emails_sender_type.xlsx")

Processing completed!
Updated file saved as emails_sender_type.xlsx


To get is_action_required

In [4]:
import pandas as pd
import re

# Read updated Excel file
df = pd.read_excel('/content/emails_sender_type.xlsx')

# Your name
my_name = "Prasanna"

# Greeting patterns containing your name
greeting_pattern = re.compile(
    rf'^\s*(dear|hi|hello|respected)\s+{re.escape(my_name)}\b\s*[,!:.-]?',
    re.IGNORECASE
)

# Check action required
def check_action_required(row):

    # External emails are always False
    if str(row['sender_type']).lower() != 'internal':
        return False

    body = str(row['body'])

    # Check whether email starts with a greeting addressing Batman
    return bool(greeting_pattern.search(body))

# Add new column
df['is_action_required'] = df.apply(check_action_required, axis=1)

# Save final Excel
df.to_excel('/content/emails_action_required.xlsx', index=False)

print("Processing completed!")
print(df[['sender_type', 'is_action_required']].head())

Processing completed!
  sender_type  is_action_required
0    internal               False
1    external               False
2    external               False
3    external               False
4    internal               False


To get labels_to_remove

In [5]:
import pandas as pd

df = pd.read_excel("/content/emails_action_required.xlsx")

df["labels_to_remove"] = df["sender_type"].apply(
    lambda x: [] if str(x).strip().lower() == "internal" else ["INBOX"]
)

df.to_excel("/content/emails_remove_labels.xlsx", index=False)

view the distinct external senders

In [7]:
import pandas as pd

# Read the original Excel
df = pd.read_excel("/content/emails_remove_labels.xlsx")

# Filter external senders
external_df = df[
    df["sender_type"].str.strip().str.lower() != "internal"
]

# Get distinct external senders
distinct_external_senders = external_df.drop_duplicates(
    subset=["from"]
)

# Save to a separate Excel file
distinct_external_senders.to_excel(
    "/content/external_senders.xlsx",
    index=False
)

print(f"Total distinct external senders: {len(distinct_external_senders)}")
print("Saved to /content/external_senders.xlsx")

Total distinct external senders: 127
Saved to /content/external_senders.xlsx


newsletter category

In [10]:
import pandas as pd

# Read Excel
df = pd.read_excel("/content/emails_remove_labels.xlsx")

# Newsletter sender email addresses (excluding the two mentioned)
newsletter_senders = {
    "mongodbteam@messages.mongodb.com",
    "noreply@medium.com",
    "newsletter@email.roadmap.sh",
    "noreply@us-education.alison.com",
    "Newsletters@yourstory.com",
    "noreply@us-news.alison.com",
    "newsletters@medium.com",
    "updates@ui.com",
    "noreply@us-skills.alison.com",
    "noreply@us-learning.alison.com",
    "community@pinecone.io",
    "noreply@us-courses.alison.com",
    "community@trypinecone.com",
    "english-quora-digest@quora.com",
    "samratzammansspace-space@quora.com",
    "ofcoursemeastrology-space@quora.com",
    "psychologicalbehaviour-space@quora.com",
    "ofcoursemebypooja-space@quora.com",
    "amen@yourstory.com",
    "info@yourstory.com",
    "thethingsworthsharing-space@quora.com",
    "updates@groq.co",
    "psychopathlevel-space@quora.com",
    "manishasinghsspace131-space@quora.com",
    "hello@portswigger.net",
    "developer@groq.co",
    "hello@mail.wispr.ai",
    "roadmap@email.roadmap.sh",
    "info@groq.co",
    "mailer@us.khoros-mail.com",
    "aamdaniatthanikharcharupiya-space@quora.com",
    "wix-team@team.wix.com",
    "healthyfye-space@quora.com",
    "husbandsandwives-space@quora.com"
}

# Normalize sender email addresses
df["from_normalized"] = (
    df["from"]
    .astype(str)
    .str.extract(r'<([^>]+)>')[0]
    .fillna(df["from"].astype(str))
    .str.strip()
    .str.lower()
)

newsletter_senders = {email.lower() for email in newsletter_senders}

# Add labels
df["labels_to_add"] = df["from_normalized"].apply(
    lambda email: ["Newsletter"] if email in newsletter_senders else []
)

# Remove helper column
df.drop(columns=["from_normalized"], inplace=True)

# Save output
df.to_excel("/content/newsletter_classification.xlsx", index=False)

print("Newsletter labeling completed!")
print("Total newsletter emails:", sum(
    1 for labels in df["labels_to_add"] if labels == ["Newsletter"]
))
print("Saved to /content/newsletter_classification.xlsx")

Newsletter labeling completed!
Total newsletter emails: 658
Saved to /content/newsletter_classification.xlsx


In [15]:

import pandas as pd
import ast
import re

# Read Excel
df = pd.read_excel("/content/newsletter_classification.xlsx")

# Normalize existing labels
def parse_labels(value):
    if isinstance(value, str):
        try:
            return ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return []
    return value if isinstance(value, list) else []

df["labels_to_add"] = df["labels_to_add"].apply(parse_labels)

# Sender email addresses
azure_devops = "azuredevops@microsoft.com"

community_senders = {
    "hello@mail.langchain.com",
    "hello@render.com",
    "notifications@openrouter.ai"
}

teams_sender = "no-reply@teams.mail.microsoft"


# Extract email address from From column
def extract_email(sender):
    if pd.isna(sender):
        return ""

    match = re.search(r'<([^>]+)>', str(sender))

    if match:
        return match.group(1).strip().lower()

    return str(sender).strip().lower()


# Classification function
def classify_email(row):

    sender = extract_email(row["from"])
    body = str(row["body"]).lower()

    # 1. Azure DevOps
    if sender == azure_devops:

        if re.search(r'build[\s_-]+succeed(?:ed)?', body):
            return ["Devops/BuildSuccess"]

        elif re.search(r'build[\s_-]+(?:failure|failed|error)', body):
            return ["Devops/BuildFailed"]

        else:
            return ["Devops/Updates"]

    # 2. Community updates
    elif sender in community_senders:
        return ["Community/updates"]

    # 3. Microsoft Teams
    elif sender == teams_sender:
        return ["Updates"]

    # Other senders: preserve existing labels
    return row["labels_to_add"]


# Apply classification
df["labels_to_add"] = df.apply(classify_email, axis=1)


# Display classification summary
print("=" * 60)
print("EMAIL CLASSIFICATION SUMMARY")
print("=" * 60)

print("\nLabel counts:")

label_counts = df["labels_to_add"].explode().value_counts()
display(label_counts.to_frame("email_count"))


# Display classified sender samples
print("\nCLASSIFIED EMAIL SAMPLES")
print("=" * 60)

classified_df = df[
    df["labels_to_add"].apply(
        lambda x: any(
            label in [
                "Devops/BuildSuccess",
                "Devops/BuildFailed",
                "Devops/Updates",
                "Community/updates",
                "Updates"
            ]
            for label in x
        )
    )
]

display(
    classified_df[
        ["from", "subject", "labels_to_add"]
    ].head(20)
)


# Save output
output_file = "/content/automated_email_classification.xlsx"

df.to_excel(output_file, index=False)

print("\nClassification completed!")
print(f"Total classified emails: {len(classified_df)}")
print(f"Output saved to: {output_file}")


EMAIL CLASSIFICATION SUMMARY

Label counts:


,email_count
labels_to_add,
Newsletter,658
Devops/BuildSuccess,325
Devops/Updates,288
Updates,86
Devops/BuildFailed,15
Community/updates,9



CLASSIFIED EMAIL SAMPLES


,from,subject,labels_to_add
5,Azure DevOps <azuredevops@microsoft.com>,[Build succeeded] Prod-Master-UI-CI - AICL-Mas...,[Devops/BuildSuccess]
6,Azure DevOps <azuredevops@microsoft.com>,[Pre-deployment approval pending] Prod-Master-...,[Devops/Updates]
7,Azure DevOps <azuredevops@microsoft.com>,[Pre-deployment approval pending] Stage-Master...,[Devops/Updates]
8,Azure DevOps <azuredevops@microsoft.com>,[Build succeeded] Stage-Master-UI-CI - AICL-Ma...,[Devops/BuildSuccess]
9,LangChain <hello@mail.langchain.com>,Your traces are flowing in: here's what's next,[Community/updates]
10,Azure DevOps <azuredevops@microsoft.com>,[Build failed] Stage-Master-UI-CI - AICL-Maste...,[Devops/BuildFailed]
11,Azure DevOps <azuredevops@microsoft.com>,[Build succeeded] Prod-Master-API-CI-CD - AICL...,[Devops/BuildSuccess]
12,Azure DevOps <azuredevops@microsoft.com>,[Build succeeded] Stage-Master-API-CI-CD - AIC...,[Devops/BuildSuccess]
13,Azure DevOps <azuredevops@microsoft.com>,[Build succeeded] Stage-Master-API-CI-CD - AIC...,[Devops/BuildSuccess]
14,Azure DevOps <azuredevops@microsoft.com>,[Build succeeded] Stage-Website-API-CI - Arise...,[Devops/BuildSuccess]



Classification completed!
Total classified emails: 723
Output saved to: /content/automated_email_classification.xlsx


In [17]:

import pandas as pd
import ast
import re
from IPython.display import display

# Read the last saved Excel file
df = pd.read_excel("/content/automated_email_classification.xlsx")

# Normalize existing labels
def parse_labels(value):
    if isinstance(value, str):
        try:
            return ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return []
    return value if isinstance(value, list) else []

df["labels_to_add"] = df["labels_to_add"].apply(parse_labels)


# Sender email addresses
azure_updates_sender = "azure-noreply@microsoft.com"

newsletter_senders = {
    "welcome@openrouter.ai",
    "no-reply@render.com"
}

azure_promotions_sender = "azure@promomail.microsoft.com"


# Extract sender email
def extract_email(sender):
    if pd.isna(sender):
        return ""

    match = re.search(r'<([^>]+)>', str(sender))

    if match:
        return match.group(1).strip().lower()

    return str(sender).strip().lower()


# Classification function
def classify_email(row):

    sender = extract_email(row["from"])

    # 1. Microsoft Azure Updates
    if sender == azure_updates_sender:
        return ["Azure/Updates"]

    # 2. OpenRouter and Render Newsletters
    elif sender in newsletter_senders:
        return ["Newsletters"]

    # 3. Microsoft Azure Promotions
    elif sender == azure_promotions_sender:
        return ["Azure/Promotions"]

    # Preserve existing labels for other senders
    return row["labels_to_add"]


# Apply classification
df["labels_to_add"] = df.apply(classify_email, axis=1)


# Display classification summary
print("=" * 60)
print("NEW EMAIL CLASSIFICATION SUMMARY")
print("=" * 60)

label_counts = df["labels_to_add"].explode().value_counts()

display(label_counts.to_frame("email_count"))


# Display matching emails
new_labels = [
    "Azure/Updates",
    "Newsletters",
    "Azure/Promotions"
]

classified_df = df[
    df["labels_to_add"].apply(
        lambda labels: any(label in new_labels for label in labels)
    )
]

print("\nCLASSIFIED EMAIL SAMPLES")
display(
    classified_df[["from", "subject", "labels_to_add"]].head(30)
)


# Save updated Excel
output_file = "/content/azure_newsletter_classification.xlsx"

df.to_excel(output_file, index=False)

print("\nClassification completed!")
print(f"Total matching emails: {len(classified_df)}")
print(f"Saved to: {output_file}")


NEW EMAIL CLASSIFICATION SUMMARY


,email_count
labels_to_add,
Newsletter,658
Devops/BuildSuccess,325
Devops/Updates,288
Updates,86
Azure/Updates,24
Devops/BuildFailed,15
Azure/Promotions,11
Community/updates,9
Newsletters,4



CLASSIFIED EMAIL SAMPLES


,from,subject,labels_to_add
3,Microsoft Azure <azure-noreply@microsoft.com>,SN 59116 Action recommended: Upgrade your App ...,[Azure/Updates]
28,OpenRouter Team <welcome@openrouter.ai>,What developers route to this week,[Newsletters]
36,Render <no-reply@render.com>,Update to Render's list of subprocessors,[Newsletters]
43,Microsoft Azure <Azure@promomail.microsoft.com>,Save up to 35% on database workloads with a sa...,[Azure/Promotions]
89,Microsoft Azure <azure-noreply@microsoft.com>,"Retirement notice: Azure OpenAI GPT-4o, GPT-4o...",[Azure/Updates]
97,Microsoft Azure <Azure@promomail.microsoft.com>,Learn to build and scale apps on Azure Cosmos DB,[Azure/Promotions]
141,OpenRouter Team <welcome@openrouter.ai>,Benchmark models on your own prompts,[Newsletters]
145,OpenRouter Team <welcome@openrouter.ai>,"Paste this, get a response",[Newsletters]
153,Microsoft Azure <azure-noreply@microsoft.com>,Migrate your .NET apps in Azure Functions to t...,[Azure/Updates]
167,Microsoft Azure <Azure@promomail.microsoft.com>,Build production-ready AI apps faster with Azu...,[Azure/Promotions]



Classification completed!
Total matching emails: 39
Saved to: /content/azure_newsletter_classification.xlsx


In [19]:

import pandas as pd
import ast
import re
from IPython.display import display

# Read last saved Excel
df = pd.read_excel("/content/azure_newsletter_classification.xlsx")

# Normalize existing labels
def parse_labels(value):
    if isinstance(value, str):
        try:
            return ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return []
    return value if isinstance(value, list) else []

df["labels_to_add"] = df["labels_to_add"].apply(parse_labels)

# Sender email addresses
adrenalin_sender = "hrms-noreply@adrenalin.in"

newsletter_senders = {
    "mongodb@team.mongodb.com",
    "mongodb-account@mongodb.com",
    "no-reply@contact.elevenlabs.io"
}

azure_updates_sender = "azure-noreply@microsoft.com"


# Extract sender email
def extract_email(sender):
    if pd.isna(sender):
        return ""

    match = re.search(r'<([^>]+)>', str(sender))

    if match:
        return match.group(1).strip().lower()

    return str(sender).strip().lower()


# Classification function
def classify_email(row):

    sender = extract_email(row["from"])

    # 1. Adrenalin
    if sender == adrenalin_sender:
        return ["Adrenalin"]

    # 2. MongoDB and ElevenLabs
    elif sender in newsletter_senders:
        return ["Newsletter"]

    # 3. Microsoft Azure Updates
    elif sender == azure_updates_sender:
        return ["Azure/Updates"]

    # Preserve existing labels for other senders
    return row["labels_to_add"]


# Apply classification
df["labels_to_add"] = df.apply(classify_email, axis=1)


# Display classification summary
print("=" * 60)
print("EMAIL CLASSIFICATION SUMMARY")
print("=" * 60)

label_counts = df["labels_to_add"].explode().value_counts()
display(label_counts.to_frame("email_count"))


# Display matching emails
new_labels = ["Adrenalin", "Newsletter", "Azure/Updates"]

classified_df = df[
    df["labels_to_add"].apply(
        lambda labels: any(label in new_labels for label in labels)
    )
]

print("\nCLASSIFIED EMAIL SAMPLES")
display(
    classified_df[["from", "subject", "labels_to_add"]].head(30)
)


# Save updated Excel
output_file = "/content/adrenalin_mongodb_classification.xlsx"

df.to_excel(output_file, index=False)

print("\nClassification completed!")
print(f"Total matching emails: {len(classified_df)}")
print(f"Saved to: {output_file}")

EMAIL CLASSIFICATION SUMMARY


,email_count
labels_to_add,
Newsletter,679
Devops/BuildSuccess,325
Devops/Updates,288
Updates,86
Azure/Updates,24
Devops/BuildFailed,15
Adrenalin,11
Azure/Promotions,11
Community/updates,9



CLASSIFIED EMAIL SAMPLES


,from,subject,labels_to_add
1,MongoDB <mongodbteam@messages.mongodb.com>,Learn more about MongoDB 9.0 - the best versio...,[Newsletter]
2,Medium Daily Digest <noreply@medium.com>,AI Agents (07): Agentic Harness (Frameworks) |...,[Newsletter]
3,Microsoft Azure <azure-noreply@microsoft.com>,SN 59116 Action recommended: Upgrade your App ...,[Azure/Updates]
15,Medium Daily Digest <noreply@medium.com>,"Why Your AI Always Picks 7, and Why Vague Prom...",[Newsletter]
18,Medium Daily Digest <noreply@medium.com>,"A Super Lightweight Alternative to VS Code, Un...",[Newsletter]
19,Kamran Ahmed <newsletter@email.roadmap.sh>,"Codex, Cloud Computing, and a new course",[Newsletter]
20,Alison Courses <noreply@us-education.alison.com>,"Prasanna Venkatesh, want to experience the wor...",[Newsletter]
21,Medium Daily Digest <noreply@medium.com>,"Stop Mixing CDM, Ontology, and Knowledge Graph...",[Newsletter]
25,Team YourStory <Newsletters@yourstory.com>,OpenAI’s Control Room can stop a model in 35 s...,[Newsletter]
31,Medium Daily Digest <noreply@medium.com>,I Tried 23 Viral Claude Skills. These Are the ...,[Newsletter]



Classification completed!
Total matching emails: 714
Saved to: /content/adrenalin_mongodb_classification.xlsx


In [22]:

import pandas as pd
import ast
import re
from IPython.display import display

# Read last saved Excel
df = pd.read_excel("/content/adrenalin_mongodb_classification.xlsx")

# Normalize existing labels
def parse_labels(value):
    if isinstance(value, str):
        try:
            return ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return []
    return value if isinstance(value, list) else []

df["labels_to_add"] = df["labels_to_add"].apply(parse_labels)

# Newsletter senders
newsletter_senders = {
    "no-reply@docker.com",
    "marc@updates.langfuse.com",
    "devcompute@msemail.microsoft.com"
}

# Lenovo senders
lenovo_senders = {
    "noreply3_iss@lenovo.com",
    "noreply1@lenovo.com"
}

# Extract sender email
def extract_email(sender):
    if pd.isna(sender):
        return ""

    match = re.search(r'<([^>]+)>', str(sender))

    if match:
        return match.group(1).strip().lower()

    return str(sender).strip().lower()


# Classification function
def classify_email(row):

    sender = extract_email(row["from"])

    # 1. Newsletter
    if sender in newsletter_senders:
        return ["Newsletter"]

    # 2. Lenovo Repair Service
    elif sender in lenovo_senders:
        return ["Lenovo/RepairService"]

    # Preserve existing labels for other senders
    return row["labels_to_add"]


# Apply classification
df["labels_to_add"] = df.apply(classify_email, axis=1)


# Display classification summary
print("=" * 60)
print("EMAIL CLASSIFICATION SUMMARY")
print("=" * 60)

label_counts = df["labels_to_add"].explode().value_counts()
display(label_counts.to_frame("email_count"))


# Display newly classified emails
new_labels = ["Newsletter", "Lenovo/RepairService"]

classified_df = df[
    df["labels_to_add"].apply(
        lambda labels: any(label in new_labels for label in labels)
    )
]

print("\nCLASSIFIED EMAIL SAMPLES")
display(
    classified_df[["from", "subject", "labels_to_add"]].tail(30)
)


# Save updated Excel
output_file = "/content/lenovo_newsletter_classification.xlsx"

df.to_excel(output_file, index=False)

print("\nClassification completed!")
print(f"Total matching emails: {len(classified_df)}")
print(f"Saved to: {output_file}")

EMAIL CLASSIFICATION SUMMARY


,email_count
labels_to_add,
Newsletter,707
Devops/BuildSuccess,325
Devops/Updates,288
Updates,86
Azure/Updates,24
Lenovo/RepairService,21
Devops/BuildFailed,15
Adrenalin,11
Azure/Promotions,11



CLASSIFIED EMAIL SAMPLES


,from,subject,labels_to_add
1935,Quora Digest <english-quora-digest@quora.com>,How to become dangerous using dark psychology.,[Newsletter]
1936,Quora Digest <english-quora-digest@quora.com>,BODY LANGUAGE TRICKS,[Newsletter]
1938,Quora Digest <english-quora-digest@quora.com>,Mental health (psychology):,[Newsletter]
1939,Quora Suggested Spaces <husbandsandwives-space...,I am leaving my husband because I don’t love h...,[Newsletter]
1942,Medium Daily Digest <noreply@medium.com>,I Deleted Notion and Obsidian. Here’s What Rep...,[Newsletter]
1943,Alison Courses <noreply@us-education.alison.com>,"Prasanna Venkatesh, learn career-transforming ...",[Newsletter]
1944,The Medium Newsletter <newsletters@medium.com>,Which stories get told?,[Newsletter]
1945,Quora Digest <english-quora-digest@quora.com>,3 Rules To Attract Your DREAM Girl (MUST READ)...,[Newsletter]
1953,Quora Digest <english-quora-digest@quora.com>,BIG NEWS : 🇮🇷 Iran has finally reacted and bel...,[Newsletter]
1954,Lenovo_Services <noreply3_iss@lenovo.com>,Service Order Completed,[Lenovo/RepairService]



Classification completed!
Total matching emails: 728
Saved to: /content/lenovo_newsletter_classification.xlsx


In [24]:

import pandas as pd
import ast
import re
from IPython.display import display

# Read last saved Excel
df = pd.read_excel("/content/lenovo_newsletter_classification.xlsx")

# Normalize existing labels
def parse_labels(value):
    if isinstance(value, str):
        try:
            return ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return []
    return value if isinstance(value, list) else []

df["labels_to_add"] = df["labels_to_add"].apply(parse_labels)


# Extract sender email
def extract_email(sender):
    if pd.isna(sender):
        return ""

    match = re.search(r'<([^>]+)>', str(sender))

    if match:
        return match.group(1).strip().lower()

    return str(sender).strip().lower()


# Classification function
def classify_email(row):

    sender = extract_email(row["from"])

    # 1. Google Updates
    if sender == "no-reply@accounts.google.com":
        return ["Google/Updates"]

    # 2. MSG91
    elif sender == "no-reply@msg91.com":
        return ["MSG91"]

    # 3. CraftSilicon
    elif sender.endswith("@craftsilicon.com"):
        return ["CraftSilicon"]

    # 4. Google Play Apps & Games
    elif sender == "playpartners-noreply@google.com":
        return ["Playstore/Updates"]

    # 5. IndiaAI Newsletter
    elif sender == "support-indiaai@indiaai.gov.in":
        return ["Newsletter"]

    # 6. Lenovo Service
    elif sender == "listening@lenovo.com":
        return ["Lenovo/Service"]

    # 7. Microsoft Tech Support
    elif sender.endswith("@techsupport.microsoft.com"):
        return ["Azure/Support"]

    # 8. ElevenLabs Newsletter
    elif sender == "developers@contact.elevenlabs.io":
        return ["Newsletter"]

    # 9. Google Play Console
    elif sender == "noreply-play-console@google.com":
        return ["Google Play Console"]

    # Preserve existing labels for other senders
    return row["labels_to_add"]


# Apply classifications
df["labels_to_add"] = df.apply(classify_email, axis=1)


# Update sender_type for CraftSilicon
craftsilicon_mask = df["from"].apply(
    lambda x: extract_email(x).endswith("@craftsilicon.com")
)

df.loc[craftsilicon_mask, "sender_type"] = "internal"


# Display classification summary
print("=" * 65)
print("EMAIL CLASSIFICATION SUMMARY")
print("=" * 65)

label_counts = df["labels_to_add"].explode().value_counts()
display(label_counts.to_frame("email_count"))


# Display newly classified emails
new_labels = [
    "Google/Updates",
    "MSG91",
    "CraftSilicon",
    "Playstore/Updates",
    "Newsletter",
    "Lenovo/Service",
    "Azure/Support",
    "Google Play Console"
]

classified_df = df[
    df["labels_to_add"].apply(
        lambda labels: any(label in new_labels for label in labels)
    )
]

print("\nCLASSIFIED EMAIL SAMPLES")
display(
    classified_df[
        ["from", "subject", "sender_type", "labels_to_add"]
    ].head(50)
)


# Display CraftSilicon internal sender details
print("\nCRAFTSILICON INTERNAL EMAILS")
display(
    df[craftsilicon_mask][
        ["from", "subject", "sender_type", "labels_to_add"]
    ]
)


# Save updated Excel
output_file = "/content/google_lenovo_support_classification.xlsx"

df.to_excel(output_file, index=False)

print("\nClassification completed!")
print(f"Total matching emails: {len(classified_df)}")
print(f"CraftSilicon emails marked internal: {craftsilicon_mask.sum()}")
print(f"Saved to: {output_file}")

EMAIL CLASSIFICATION SUMMARY


,email_count
labels_to_add,
Newsletter,714
Devops/BuildSuccess,325
Devops/Updates,288
Updates,86
Azure/Updates,24
Lenovo/RepairService,21
Devops/BuildFailed,15
CraftSilicon,14
Azure/Support,12



CLASSIFIED EMAIL SAMPLES


,from,subject,sender_type,labels_to_add
1,MongoDB <mongodbteam@messages.mongodb.com>,Learn more about MongoDB 9.0 - the best versio...,external,[Newsletter]
2,Medium Daily Digest <noreply@medium.com>,AI Agents (07): Agentic Harness (Frameworks) |...,external,[Newsletter]
15,Medium Daily Digest <noreply@medium.com>,"Why Your AI Always Picks 7, and Why Vague Prom...",external,[Newsletter]
18,Medium Daily Digest <noreply@medium.com>,"A Super Lightweight Alternative to VS Code, Un...",external,[Newsletter]
19,Kamran Ahmed <newsletter@email.roadmap.sh>,"Codex, Cloud Computing, and a new course",external,[Newsletter]
20,Alison Courses <noreply@us-education.alison.com>,"Prasanna Venkatesh, want to experience the wor...",external,[Newsletter]
21,Medium Daily Digest <noreply@medium.com>,"Stop Mixing CDM, Ontology, and Knowledge Graph...",external,[Newsletter]
25,Team YourStory <Newsletters@yourstory.com>,OpenAI’s Control Room can stop a model in 35 s...,external,[Newsletter]
31,Medium Daily Digest <noreply@medium.com>,I Tried 23 Viral Claude Skills. These Are the ...,external,[Newsletter]
33,Alison Courses <noreply@us-education.alison.com>,"Prasanna Venkatesh, today's a good day to save...",external,[Newsletter]



CRAFTSILICON INTERNAL EMAILS


,from,subject,sender_type,labels_to_add
318,"""sunil.m"" <sunil.m@craftsilicon.com>",RE: 14630 - BRConnect Implementation,internal,[CraftSilicon]
434,<vikash.kumar@craftsilicon.com>,RE: Customer App Migration.,internal,[CraftSilicon]
442,<vikash.kumar@craftsilicon.com>,RE: Customer App Migration.,internal,[CraftSilicon]
448,"""sunil.m"" <sunil.m@craftsilicon.com>",RE: 14630 - BRConnect Implementation,internal,[CraftSilicon]
451,<vikash.kumar@craftsilicon.com>,RE: Customer App Migration.,internal,[CraftSilicon]
453,<vikash.kumar@craftsilicon.com>,RE: Customer App Migration.,internal,[CraftSilicon]
455,<vikash.kumar@craftsilicon.com>,RE: Customer App Migration.,internal,[CraftSilicon]
498,"""sunil.m"" <sunil.m@craftsilicon.com>",RE: 14630 - BRConnect Implementation,internal,[CraftSilicon]
573,jyoti <jyoti.annarao@craftsilicon.com>,Re: 14630 - BRConnect Implementation,internal,[CraftSilicon]
602,"""sunil.m"" <sunil.m@craftsilicon.com>",RE: 14630 - BRConnect Implementation,internal,[CraftSilicon]



Classification completed!
Total matching emails: 767
CraftSilicon emails marked internal: 14
Saved to: /content/google_lenovo_support_classification.xlsx


Viewing unclassified records

In [25]:
import pandas as pd
import ast
from IPython.display import display

# Read the classified Excel file
df = pd.read_excel("/content/google_lenovo_support_classification.xlsx")

# Convert labels_to_add column into lists
df["labels_to_add"] = df["labels_to_add"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

# Filter only external senders
external_df = df[
    df["sender_type"].astype(str).str.strip().str.lower() != "internal"
]

# Filter external records where labels_to_add is empty
unlabeled_df = external_df[
    external_df["labels_to_add"].apply(lambda x: x == [])
]

# Get distinct external sender details
distinct_unlabeled_senders = unlabeled_df[
    ["from", "sender_type"]
].drop_duplicates(subset=["from"])

# Display summary
print("=" * 70)
print("EXTERNAL EMAIL CLASSIFICATION SUMMARY")
print("=" * 70)

print(f"\nTotal external email records: {len(external_df)}")
print(f"External records with labels_to_add = []: {len(unlabeled_df)}")
print(f"Distinct external senders with labels_to_add = []: {len(distinct_unlabeled_senders)}")

# Display 10 sample distinct senders
print("\n10 SAMPLE DISTINCT EXTERNAL SENDERS")
print("=" * 70)
display(distinct_unlabeled_senders.head(10))

# Display all distinct external senders
print("\nALL DISTINCT EXTERNAL SENDERS")
print("=" * 70)
display(distinct_unlabeled_senders)

# Sender frequency - highest to lowest
print("\nEXTERNAL SENDER EMAIL COUNT RANKING")
print("=" * 70)

sender_frequency = (
    unlabeled_df["from"]
    .value_counts()
    .reset_index()
)

sender_frequency.columns = ["from", "email_count"]

sender_frequency.index = sender_frequency.index + 1
sender_frequency.index.name = "rank"

display(sender_frequency)

# Display top 10 highest email senders
print("\nTOP 10 SENDERS WITH HIGHEST EMAIL COUNT")
print("=" * 70)

display(sender_frequency.head(10))

# Display highest sender
if not sender_frequency.empty:
    top_sender = sender_frequency.iloc[0]

    print("\nHIGHEST EMAIL SENDER")
    print(f"Sender: {top_sender['from']}")
    print(f"Total emails: {top_sender['email_count']}")

EXTERNAL EMAIL CLASSIFICATION SUMMARY

Total external email records: 1603
External records with labels_to_add = []: 56
Distinct external senders with labels_to_add = []: 42

10 SAMPLE DISTINCT EXTERNAL SENDERS


,from,sender_type
72,Google Developers <googledevelopers-noreply@go...,external
91,uxresearch@mongodb.com,external
102,The Render team <dx@render.com>,external
128,Microsoft Invitations on behalf of PSCCA <MSSe...,external
131,Google Cloud <googlecloud@google.com>,external
135,Tavily team <jakki.jakaj@tavily.com>,external
142,Vercel <registration@vercel.com>,external
147,Google AI Studio <googleaistudio-noreply@googl...,external
280,Outskill <no-reply@zoom.us>,external
313,The Google Workspace Team <workspace-noreply@g...,external



ALL DISTINCT EXTERNAL SENDERS


,from,sender_type
72,Google Developers <googledevelopers-noreply@go...,external
91,uxresearch@mongodb.com,external
102,The Render team <dx@render.com>,external
128,Microsoft Invitations on behalf of PSCCA <MSSe...,external
131,Google Cloud <googlecloud@google.com>,external
135,Tavily team <jakki.jakaj@tavily.com>,external
142,Vercel <registration@vercel.com>,external
147,Google AI Studio <googleaistudio-noreply@googl...,external
280,Outskill <no-reply@zoom.us>,external
313,The Google Workspace Team <workspace-noreply@g...,external



EXTERNAL SENDER EMAIL COUNT RANKING


,from,email_count
rank,,
1,Google Developers <googledevelopers-noreply@go...,2
2,Microsoft Invitations on behalf of PSCCA <MSSe...,2
3,Outskill <no-reply@zoom.us>,2
4,Microsoft 365 Copilot <m365copilotupdates@micr...,2
5,Lucid <email@e.lucid.co>,2
6,MSG91 <partners@msg91.com>,2
7,googleplay-developer-support@google.com,2
8,Microsoft on behalf of ARISECAPITAL <msonlines...,2
9,Sumedh Dubey <sumedhdubey@microsoft.com>,2



TOP 10 SENDERS WITH HIGHEST EMAIL COUNT


,from,email_count
rank,,
1,Google Developers <googledevelopers-noreply@go...,2
2,Microsoft Invitations on behalf of PSCCA <MSSe...,2
3,Outskill <no-reply@zoom.us>,2
4,Microsoft 365 Copilot <m365copilotupdates@micr...,2
5,Lucid <email@e.lucid.co>,2
6,MSG91 <partners@msg91.com>,2
7,googleplay-developer-support@google.com,2
8,Microsoft on behalf of ARISECAPITAL <msonlines...,2
9,Sumedh Dubey <sumedhdubey@microsoft.com>,2



HIGHEST EMAIL SENDER
Sender: Google Developers <googledevelopers-noreply@google.com>
Total emails: 2
